# UTS IN0417 — Kecerdasan Buatan dan ML Dasar
**Mini-Project Individual** - Binary Classification: Bank Note Authentication

---



**Nama:** DICA DAMAR BINTANG  

**NIM:** 261635003

**Paket Soal:** G

**Dosen:** Kodrat Mahatma, S.T., M.T.

**random_state = 3**

Identitas: DICA DAMAR BINTANG, NIM 261635003. Tiga digit `003` dikonversi menjadi integer `3`, bukan bilangan desimal.

**Tujuan:** membandingkan k-NN dan Decision Tree untuk prediksi kelas banknote pada test set yang sama. Target adalah kelas diskret, bukan nilai uang atau citra mentah.

## 1. Import library dan konfigurasi

In [ ]:
# Mengimpor alat untuk mengelola file, mengolah data, membuat grafik,
# membagi data, melatih model, dan mengevaluasi hasilnya.
from pathlib import Path
import platform
import hashlib
import io
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score, classification_report)

# Identitas mahasiswa. NIM disimpan sebagai string.
NAMA = "DICA DAMAR BINTANG"
NIM = "261635003"
AKHIR_NIM = "003"
RANDOM_STATE = 3

# Folder untuk menyimpan tabel dan gambar hasil.
OUT = Path("output")
OUT.mkdir(parents=True, exist_ok=True)
print("Nama:", NAMA, "| NIM:", NIM, "| Paket G | random_state:", RANDOM_STATE)
print("Python:", platform.python_version(), "| pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)

**Penjelasan output**

Identitas, paket soal, dan random_state=3 ditampilkan bersama versi Python, pandas, dan scikit-learn. Versi ini membantu melacak lingkungan yang dipakai.

## 2. Load data nyata dari Kaggle

In [ ]:
# Menentukan lokasi file CSV dan alamat untuk mengunduh dataset.
DATA_PATH = Path("BankNote_Authentication.csv")
DOWNLOAD_URL = "https://www.kaggle.com/api/v1/datasets/download/ritesaluja/bank-note-authentication-uci-data"
# Cadangan: dataset yang sama dari UCI (tanpa header kolom).
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt"
KOLOM = ["variance", "skewness", "curtosis", "entropy", "class"]

def unduh_kaggle():
    # Kaggle mengirimkan dataset dalam bentuk ZIP; CSV perlu diekstrak dahulu.
    response = requests.get(DOWNLOAD_URL, timeout=90)
    response.raise_for_status()
    content = response.content

    if zipfile.is_zipfile(io.BytesIO(content)):
        with zipfile.ZipFile(io.BytesIO(content)) as archive:
            csv_files = [name for name in archive.namelist()
                         if name.lower().endswith(".csv")]
            if not csv_files:
                raise FileNotFoundError(
                    "Unduhan Kaggle berhasil, tetapi file CSV tidak ditemukan di dalam ZIP."
                )
            # Gunakan CSV yang namanya paling sesuai, jika tersedia.
            selected_csv = next(
                (name for name in csv_files
                 if Path(name).name.lower() == DATA_PATH.name.lower()),
                csv_files[0]
            )
            DATA_PATH.write_bytes(archive.read(selected_csv))
    elif content.lstrip().lower().startswith(b"variance"):
        # Cadangan jika server mengembalikan CSV secara langsung.
        DATA_PATH.write_bytes(content)
    else:
        # Biasanya halaman login Kaggle, bukan data.
        raise ValueError("Kaggle tidak mengembalikan file data (kemungkinan perlu login).")

def unduh_uci():
    # File UCI berisi 5 kolom tanpa nama; nama kolom ditambahkan di sini.
    response = requests.get(UCI_URL, timeout=90)
    response.raise_for_status()
    tabel = pd.read_csv(io.StringIO(response.text), header=None, names=KOLOM)
    tabel.to_csv(DATA_PATH, index=False)

if not DATA_PATH.exists():
    try:
        unduh_kaggle()
        print("Dataset diunduh dari Kaggle.")
    except Exception as err_kaggle:
        print("Unduhan Kaggle gagal:", err_kaggle)
        try:
            unduh_uci()
            print("Dataset diunduh dari UCI (isi sama dengan versi Kaggle).")
        except Exception as err_uci:
            print("Unduhan UCI gagal:", err_uci)
            try:
                # Cara terakhir di Google Colab: unggah file CSV secara manual.
                from google.colab import files
                print("Unggah file BankNote_Authentication.csv dari Kaggle.")
                files.upload()
            except ImportError:
                pass
            if not DATA_PATH.exists():
                raise FileNotFoundError(
                    "Letakkan BankNote_Authentication.csv di folder yang sama dengan notebook.")

# Membaca CSV sebagai tabel Pandas bernama raw.
raw = pd.read_csv(DATA_PATH)
assert list(raw.columns) == ["variance", "skewness", "curtosis", "entropy", "class"], (
    f"Kolom dataset tidak sesuai: {raw.columns.tolist()}"
)
assert set(raw["class"].unique()) == {0, 1}, (
    "Kolom class harus berisi label 0 dan 1."
)

print("Ukuran dataset mentah:", raw.shape)
display(raw.head(10))

**Penjelasan output**

Dataset mentah berisi 1.372 baris dan 5 kolom.

Lima baris pertama memperlihatkan empat fitur numerik dan kolom class sebagai target.

## 3. Data understanding

In [ ]:
# Memeriksa isi, tipe data, nilai kosong, duplikat, dan sebaran nilai dataset.
print("Nama kolom:", raw.columns.tolist())
print("Target: class, klasifikasi biner; dtype penyimpanan:", raw["class"].dtype)
display(pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum()}))
print("Duplikat baris lengkap:", raw.duplicated().sum())
display(raw.describe().T)  # Menampilkan statistik kolom numerik; T menukar baris dan kolom.
display(raw["class"].value_counts().sort_index().rename_axis("class").to_frame("jumlah_mentah"))

**Penjelasan output**

Daftar kolom menunjukkan bahwa class adalah target klasifikasi biner yang disimpan sebagai bilangan bulat.

Keempat fitur bertipe desimal dan tidak ada nilai kosong di semua kolom.

Ada 24 baris duplikat di luar kemunculan pertama masing-masing baris.

Statistik seperti median, simpangan baku, dan nilai minimum–maksimum membantu melihat sebaran tiap fitur.

Data mentah berisi 762 baris kelas 0 dan 610 baris kelas 1.

## 4. Preprocessing


In [ ]:
# Menyiapkan fitur, memeriksa kualitas data, dan menghapus baris duplikat.
FEATURES = ["variance", "skewness", "curtosis", "entropy"]
assert raw.isna().sum().sum() == 0, "Data berubah: tangani missing value sebelum melanjutkan."
assert np.isfinite(raw[FEATURES].to_numpy()).all()
conflicts = raw.groupby(FEATURES)["class"].nunique().gt(1).sum()
print("Kombinasi fitur dengan konflik label:", conflicts)
assert conflicts == 0
# Index asli tetap dipertahankan untuk melacak prediksi ke CSV mentah.
data = raw.drop_duplicates().copy()
X, y = data[FEATURES], data["class"].astype(int)
print("Baris sebelum:", len(raw), "| dihapus:", len(raw)-len(data), "| sesudah:", len(data))
class_counts = y.value_counts().sort_index()
display(pd.DataFrame({"jumlah": class_counts, "persen": 100*class_counts/len(y)}))
fig, ax = plt.subplots(figsize=(6,3))
ax.bar(["0 (autentik)", "1 (palsu)"], class_counts, color=["#183956", "#00A38C"])
ax.set(ylabel="Jumlah sampel", title="Distribusi kelas setelah deduplikasi")
for i,v in enumerate(class_counts): ax.text(i,v+5,str(v),ha="center")
fig.tight_layout(); fig.savefig(OUT/"distribusi_kelas.png", dpi=180); plt.show()

**Penjelasan output**

Tidak ada kombinasi fitur dengan label yang bertentangan. Setelah 24 duplikat dibuang, tersisa 1.348 baris.
Setelah pembersihan, kelas 0 berjumlah 738 (54,75%) dan kelas 1 berjumlah 610 (45,25%).
Grafik batang memperlihatkan bahwa kelas 0 sedikit lebih banyak daripada kelas 1.

## 5. Train-test split yang sama untuk kedua model
Test 20%, `random_state=3`

In [ ]:
# Membagi data menjadi bagian pelatihan dan pengujian.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
assert set(X_train.index).isdisjoint(X_test.index)
assert not pd.concat([X_train, X_test]).duplicated().any()
print("Train:", X_train.shape, "| Test:", X_test.shape)
split_summary = pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()}).sort_index()
display(split_summary)
pd.DataFrame({"row_index_csv": data.index,
              "split": np.where(data.index.isin(X_train.index), "train", "test")}).to_csv(OUT/"split_indices.csv", index=False)

**Penjelasan output**

Data dibagi menjadi 1.078 baris latih dan 270 baris uji. Keduanya memiliki empat fitur.
Tabel menunjukkan bahwa proporsi kedua kelas tetap hampir sama di data latih dan data uji.

## 6. Modeling
**k-NN:** StandardScaler + 5 tetangga, bobot uniform, jarak Euclidean (`p=2`). Prediksi dari suara mayoritas tetangga terdekat. Scaling mencegah fitur berskala besar mendominasi jarak.

**Decision Tree:** kriteria Gini, `max_depth=5`, `min_samples_leaf=2`, `random_state=3`. Membagi ruang fitur menggunakan aturan ambang. Kedalaman dibatasi secara apriori untuk mengurangi kompleksitas; bukan hasil tuning.

Tidak mengklaim parameter ini optimal. Perbedaan skor berlaku pada satu holdout ini saja.

In [ ]:
# Membuat dua model klasifikasi.
models = {
    "k-NN": Pipeline([("scaler", StandardScaler()),
                      ("model", KNeighborsClassifier(n_neighbors=5, weights="uniform", p=2))]),
    "Decision Tree": DecisionTreeClassifier(criterion="gini", max_depth=5,
                    min_samples_leaf=2, random_state=RANDOM_STATE)
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "selesai dilatih | accuracy train:", round(model.score(X_train,y_train),6))
print("Kedalaman tree:", models["Decision Tree"].get_depth(),
      "| jumlah daun:", models["Decision Tree"].get_n_leaves())

**Penjelasan output**

Akurasi latih k-NN mencapai 99,91%, sedangkan Decision Tree 99,63%. Pohon memiliki kedalaman 5 dan 19 daun. Angka ini belum mengukur kinerja pada data uji.

## 7. Prediction


In [ ]:
# Membuat prediksi pada data uji dan menyusun tabel hasilnya.
predictions = {name: model.predict(X_test) for name,model in models.items()}
pred_table = X_test.copy()
pred_table.insert(0,"row_index_csv", X_test.index)
pred_table["actual"] = y_test
for name,pred in predictions.items():
    pred_table["pred_"+name] = pred
    pred_table["benar_"+name] = (pred == y_test.to_numpy())
pred_table.to_csv(OUT/"prediksi_test.csv", index=False)
display(pred_table.head(15))
print("Contoh kesalahan (jika ada):")
display(pred_table[(~pred_table["benar_k-NN"]) | (~pred_table["benar_Decision Tree"])])

**Penjelasan output**

Tabel menyandingkan label aktual dan prediksi kedua model.

Indeks CSV membantu menelusuri kembali baris aslinya.

Tulisan ini menandai bagian yang khusus menampilkan prediksi keliru.

Tabel ini menampilkan contoh baris yang salah diprediksi oleh minimal satu model.

Bandingkan actual dengan kolom prediksi untuk menemukan model yang keliru.

## 8. Evaluation: confusion matrix dan metrik
**Orientasi:** baris = aktual, kolom = prediksi, urutan label `[0,1]`. Maka matriks adalah `[[TN, FP], [FN, TP]]`.

Positif = kelas 1 (palsu, berdasarkan konvensi proyek):
- **TN:** autentik diprediksi autentik; **TP:** palsu diprediksi palsu.
- **FP:** autentik diprediksi palsu; transaksi sah dapat tertolak atau perlu pemeriksaan ulang.
- **FN:** palsu diprediksi autentik; berisiko diterima sebagai uang sah.

Accuracy = (TP+TN)/N; Precision = TP/(TP+FP); Recall = TP/(TP+FN); Specificity = TN/(TN+FP); F1 = 2TP/(2TP+FP+FN).

In [ ]:
# Menghitung metrik evaluasi secara manual dan dengan Scikit-learn.
cms, rows, manual_rows = {}, [], []
for name, pred in predictions.items():
    cm = confusion_matrix(y_test, pred, labels=[0,1])
    cms[name] = cm
    tn,fp,fn,tp = map(int,cm.ravel())
    # Rumus aritmetika langsung dari confusion matrix (manual).
    manual = {"Accuracy": (tp+tn)/(tp+tn+fp+fn), # Proporsi seluruh prediksi yang benar.
              "Precision": tp/(tp+fp) if tp+fp else 0., # Ketepatan prediksi kelas positif.
              "Recall": tp/(tp+fn) if tp+fn else 0., # Proporsi kelas positif yang berhasil ditemukan.
              "Specificity": tn/(tn+fp) if tn+fp else 0., # Proporsi kelas negatif yang dikenali.
              "F1": 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.}  # Gabungan precision dan recall.
    # Specificity sama dengan recall kelas negatif (pos_label=0).
    auto = {"Accuracy": accuracy_score(y_test,pred),  # Menghitung accuracy dengan Scikit-learn.
            "Precision": precision_score(y_test,pred,pos_label=1,zero_division=0),  # Precision dengan kelas 1 sebagai positif.
            "Recall": recall_score(y_test,pred,pos_label=1,zero_division=0),  # Recall kelas 1, yaitu kelas palsu.
            "Specificity": recall_score(y_test,pred,pos_label=0,zero_division=0),  # Recall kelas 0 sebagai specificity.
            "F1": f1_score(y_test,pred,pos_label=1,zero_division=0)} # F1 untuk kelas 1.
    assert np.allclose(list(manual.values()), list(auto.values()))  # Memastikan hasil manual dan Scikit-learn cocok.
    print("\n",name,"| TN, FP, FN, TP =",tn,fp,fn,tp)
    print(f"Accuracy manual = ({tp}+{tn})/({tp}+{tn}+{fp}+{fn}) = {manual['Accuracy']:.6f}")
    print(f"Recall manual = {tp}/({tp}+{fn}) = {manual['Recall']:.6f}")
    comparison = pd.DataFrame({"manual":manual,"scikit-learn":auto})
    display(comparison)
    manual_rows.append(comparison.assign(model=name).rename_axis("metric").reset_index())
    print(classification_report(y_test,pred,labels=[0,1],target_names=["0/autentik","1/palsu"],zero_division=0))
    rows.append({"Model":name, **auto,"TN":tn,"FP":fp,"FN":fn,"TP":tp,
                 "Train_accuracy": models[name].score(X_train,y_train)})
results = pd.DataFrame(rows).set_index("Model") # Menggabungkan ringkasan metrik semua model.
results.to_csv(OUT/"metrik_evaluasi.csv")  # Menyimpan metrik evaluasi ke CSV.
pd.concat(manual_rows).to_csv(OUT/"perhitungan_manual.csv",index=False)
display(results.round(6))
fig,axes = plt.subplots(1,2,figsize=(10,4)) # Membuat dua area grafik berdampingan.
for ax,(name,cm) in zip(axes,cms.items()): # Memasangkan grafik dengan matriks tiap model.
    ConfusionMatrixDisplay(cm,display_labels=["0/autentik","1/palsu"]).plot(ax=ax,cmap="Blues",colorbar=False) # Menggambar confusion matrix dengan gradasi biru.
    ax.set(title=name,xlabel="Prediksi",ylabel="Aktual")
fig.suptitle("Baris aktual, kolom prediksi | positif = 1 | [[TN, FP], [FN, TP]]")
fig.tight_layout(); fig.savefig(OUT/"confusion_matrix.png",dpi=180); plt.show()

**Penjelasan output**

1. Hitungan manual merangkum prediksi benar dan salah masing-masing model. k-NN melakukan 1 kesalahan, sedangkan Decision Tree melakukan 6.
2. Metrik k-NN dari perhitungan manual cocok dengan hasil scikit-learn. Akurasi ujinya 99,63%.
3. Laporan k-NN menunjukkan recall kelas 1 sebesar 100%: semua 122 sampel kelas 1 terdeteksi.
4. Metrik Decision Tree dari perhitungan manual juga cocok dengan hasil scikit-learn. Akurasi ujinya 97,78%.
5. Laporan Decision Tree menunjukkan kinerja per kelas. Recall kelas 1 sebesar 97,54%, artinya 119 dari 122 sampel terdeteksi.
6. Tabel ringkasan menunjukkan k-NN unggul pada akurasi, recall, dan F1 di data uji ini.
7. Pada kedua matriks, diagonal menunjukkan prediksi benar. Di luar diagonal, k-NN memiliki 1 positif palsu; Decision Tree memiliki 3 positif palsu dan 3 negatif palsu.

## 9. Interpretasi hasil dan perbandingan

In [ ]:
# Membandingkan hasil model dengan baseline dan memilih kandidat.
baseline = y_test.value_counts().max()/len(y_test)
# Menghitung akurasi jika selalu menebak kelas yang paling banyak di data test.
print(f"Baseline selalu menebak kelas mayoritas pada test: {baseline:.2%}")
for name, row in results.iterrows():
    print(f"{name}: {row['Accuracy']:.2%} accuracy, {row['Recall']:.2%} recall palsu, "
          f"{int(row['FN'])} FN, {int(row['FP'])} FP; F1 {row['F1']:.4f}.")  # Menampilkan metrik utama dan jumlah kesalahan.
# Prioritas konteks yang dinyatakan: recall positif dahulu, F1 sebagai pemecah seri.
ranked = results.sort_values(["Recall","F1"],ascending=False)   # Mengurutkan model berdasarkan recall lalu F1.
preferred = ranked.index[0]
print("Kandidat berdasarkan recall lalu F1 pada holdout ini:", preferred)
print("Ini bukan bukti superioritas umum atau hasil validasi eksternal.")
results[["Accuracy","Precision","Recall","Specificity","F1"]].T.plot.bar(
    figsize=(9,4), color=["#183956","#00A38C"],ylim=(0,1.09),rot=0)
plt.ylabel("Skor (0–1)"); plt.title("Evaluasi test, bukan hasil tuning")
plt.legend(loc="lower right"); plt.tight_layout(); plt.savefig(OUT/"perbandingan_metrik.png",dpi=180); plt.show()
summary = {"raw_rows":len(raw),"rows":len(data),"duplicates_removed":len(raw)-len(data),
 "train":len(X_train),"test":len(X_test),"class_counts":{str(k):int(v) for k,v in class_counts.items()},
 "split_counts":{str(c):{str(k):int(v) for k,v in split_summary[c].items()} for c in split_summary},
 "baseline":float(baseline),"preferred":preferred,"random_state":RANDOM_STATE,
 "results":results.reset_index().to_dict(orient="records"),
 "sha256":hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
 "versions":{"python":platform.python_version(),"pandas":pd.__version__,"sklearn":sklearn.__version__}}

**Penjelasan output**

1. Jika selalu menebak kelas mayoritas, akurasinya hanya 54,81%. Kedua model melampaui patokan ini, dan k-NN tidak melewatkan sampel kelas 1 pada data uji.
2. Grafik memudahkan perbandingan metrik kedua model. Batang k-NN sedikit lebih tinggi, tetapi hasil ini hanya berlaku untuk pembagian data uji yang digunakan.

### Interpretasi konteks, bukan hanya angka
FN lebih diprioritaskan **dengan asumsi** biaya menerima uang palsu lebih besar daripada biaya pemeriksaan ulang uang autentik. Ini pertimbangan konteks, bukan hasil estimasi biaya bisnis dari dataset. FP juga penting karena dapat menolak transaksi sah. Jika pemetaan semantik label diganti, interpretasi autentik/palsu harus disesuaikan.

Distribusi kelas tidak persis 50:50, tetapi bukan ketimpangan ekstrem. Accuracy harus dibaca bersama baseline mayoritas, recall, specificity, precision, dan F1; model yang selalu menebak kelas mayoritas dapat tampak cukup akurat meskipun gagal mengenali kelas lainnya.

## 10. Kesimpulan dan keterbatasan
Dua model berhasil dievaluasi pada split yang sama. Lihat output bagian 9 untuk kandidat model berdasarkan prioritas recall kelas positif, lalu F1. Nilai train membantu memantau gap train-test, tetapi tidak cukup untuk membuktikan tidak ada overfitting. Jika skor tinggi/sempurna, itu hanya berlaku untuk test set ini, bukan jaminan untuk uang baru.

Keterbatasan: satu holdout kecil, tanpa cross-validation, tanpa optimasi hiperparameter, tanpa data eksternal/ID uang untuk group split, dan tanpa pengukuran biaya kesalahan. Deduplikasi mencegah kebocoran salinan persis, bukan semua ketergantungan antarobservasi. Verifikasi semantik label diperlukan sebelum penggunaan operasional. Model menerima empat fitur numerik, tidak langsung mendeteksi foto uang.

**Hal yang dipelajari:** pencegahan leakage, scaling yang benar, perbandingan adil, perbedaan FP/FN, serta alasan recall tidak boleh digantikan oleh accuracy saja.
